In [ ]:
import torch
import numpy as np
import os
import xarray as xr 
import matplotlib.pyplot as plt 


In [ ]:
preds_dir = '../preds/'
dataset_dir = '../data/'

SCALING_FACTOR = 4

In [ ]:
def load_dataset(data, start, end, patch_size):
    dataset = xr.open_dataset(data).sel(valid_time=slice(start, end)).isel(
        longitude=slice(0, patch_size), latitude=slice(0, patch_size))
    return dataset

def normalize_data(data, custom_scale = None):
    returned_scale = {}
    for var in data:
        
        if custom_scale and var in custom_scale:
            min_val = custom_scale[var]['min']
            max_val = custom_scale[var]['max']
        else:
            min_val = data[var].values.min()
            max_val = data[var].values.max()
            
            returned_scale[var] = {'min': min_val, 'max': max_val}
            
            
        data[var] = (data[var] - min_val) / (max_val - min_val)
        
        
    return data, returned_scale


def average_pooling(data, scale, to_int=False):
    new_h, new_w = data.shape[0] // scale, data.shape[1] // scale
    data = data.reshape(new_h, scale, new_w, scale).mean(axis=(1, 3))
    if to_int:
        data = np.uint8(data)
    return data

def get_original_wind_speed(data, scale):
    u_min = scale['u10']['min'].item()
    u_max = scale['u10']['max'].item()
    v_min = scale['v10']['min'].item()
    v_max = scale['v10']['max'].item()
    
    norm_u = data[:,0,:,:]
    norm_v = data[:,1,:,:]
    
    u = norm_u * (u_max - u_min) + u_min
    v = norm_v * (v_max - v_min) + v_min
    print(f"norm_u range: {norm_u.min()} to {norm_u.max()}")
    print(f"norm_v range: {norm_v.min()} to {norm_v.max()}")

    print(f"u range: {u.min()} to {u.max()}, v range: {v.min()} to {v.max()}")

    return torch.sqrt(u**2 + v**2)

In [ ]:
files = os.listdir(preds_dir)
print(files)

['output_1984_4x.pt', 'output_1984_2x.pt', 'baseline_1984_2x.pt', 'baseline_1984_4x.pt']


# Ground Truth

In [ ]:
dataset = load_dataset(dataset_dir + 'era5_training.nc',
                       start='1980-01-01', 
                       end='1984-12-31', 
                       patch_size=32)

train_split = dataset.sel(valid_time=slice('1980-01-01', '1982-12-31'))
eval_split = dataset.sel(valid_time=slice('1984-01-01', '1984-12-31'))

_, scale = normalize_data(train_split)
eval_split, _ = normalize_data(eval_split, scale)
print(scale)
#gstack first two channels in a tensor+
eval_split_tensor = torch.stack([torch.tensor(eval_split[var].values) for var in eval_split.data_vars], dim=1)
print(eval_split_tensor.shape)
ground_truth_wind_speeds = np.sqrt(eval_split['u10'].values**2 + eval_split['v10'].values**2)
print(ground_truth_wind_speeds.shape)


{'u10': {'min': np.float32(-15.279355), 'max': np.float32(24.534174)}, 'v10': {'min': np.float32(-16.699005), 'max': np.float32(21.35775)}, 'd2m': {'min': np.float32(245.78934), 'max': np.float32(296.0891)}, 't2m': {'min': np.float32(248.4476), 'max': np.float32(306.05835)}, 'msl': {'min': np.float32(96446.56), 'max': np.float32(104656.41)}, 'tp': {'min': np.float32(0.0), 'max': np.float32(0.01362586)}}
torch.Size([1464, 6, 32, 32])
(1464, 32, 32)


# Low res input

In [ ]:
u10 = []
v10 = []
for i in range(eval_split['u10'].shape[0]):
    u10.append(average_pooling(eval_split['u10'][i].values, SCALING_FACTOR, False))
    v10.append(average_pooling(eval_split['v10'][i].values, SCALING_FACTOR, False))
u10 = np.array(u10)
v10 = np.array(v10)
# get wind speed
lr_wind_speeds = np.sqrt(u10**2 + v10**2)

print(lr_wind_speeds.shape)




(1464, 8, 8)


# Model Output

In [ ]:
ws_fields = []
print(SCALING_FACTOR)
for file in files:
    #get specific file name
    if file.endswith(f'{SCALING_FACTOR}x.pt'):
        
        field = torch.load(preds_dir + file, weights_only=False)
        
        if file == f'baseline_1984_{SCALING_FACTOR}x.pt':
            baseline_alt = torch.sqrt(field[:, 0, :, :]**2 + field[:, 1, :, :]**2)
            baseline = get_original_wind_speed(field, scale)
        else:
            wind_speed_fields_alt = torch.sqrt(field[:, 0, :, :]**2 + field[:, 1, :, :]**2)
            wind_speed_fields = get_original_wind_speed(field, scale)
            print(wind_speed_fields_alt.shape)
        
            
        
    ws_fields.append(wind_speed_fields_alt)
    ws_fields.extend([ ground_truth_wind_speeds, lr_wind_speeds])

4
norm_u range: -1.112481713294983 to 1.1175581216812134
norm_v range: -1.113476037979126 to 1.1321929693222046
u range: -59.5711784362793 to 29.214580535888672, v range: -59.07429122924805 to 26.38858413696289
torch.Size([1464, 32, 32])
norm_u range: -0.009438526816666126 to 0.871938169002533
norm_v range: 0.06692583113908768 to 0.964139461517334
u range: -15.655136108398438 to 19.43558120727539, v range: -14.15202522277832 to 19.99301528930664


# Compare 


In [ ]:
names = ["EDSR", "Bilinear", "High resolution Ground Truth", "Low resolution"]    

In [ ]:
"""
fig, ax = plt.subplots(1, len(names), figsize=(20, 5))
index = np.random.randint(0, ws_fields[0].shape[0])
# get field at index for each field in ws_fields
ws_fields = [field[index,:,:] for field in ws_fields]
print(len(ws_fields))
# Ensure ax is always a list, even if there's only one subplot
if len(names) == 1:
    ax = [ax]

for i, ws_field in enumerate(ws_fields):
    ax[i].imshow(ws_field, cmap='inferno')
    ax[i].set_title(names[i])
    ax[i].axis('off')

plt.show()
"""


fig, ax = plt.subplots(1, len(names), figsize=(20, 5))
index = np.random.randint(0, ws_fields[0].shape[0])
# get field at index for each field in ws_fields
ws_fields = [field[index,:,:] for field in ws_fields]
print(len(ws_fields))
# Ensure ax is always a list, even if there's only one subplot
if len(names) == 1:
    ax = [ax]

for i, ws_field in enumerate(ws_fields):
    ax[i].imshow(ws_field, cmap='inferno')
    ax[i].set_title(names[i])
    ax[i].axis('off')

plt.show()



In [ ]:
def plot_comparison(gt, lr, sr, baseline, index):
    fig, ax = plt.subplots(1, 4, figsize=(12, 3))
    ax[0].imshow(gt[index], cmap='inferno')
    ax[0].set_title('Ground Truth')
    ax[1].imshow(lr[index], cmap='inferno')
    ax[1].set_title('Low Resolution')
    ax[2].imshow(sr[index].detach().numpy(), cmap='inferno')
    ax[2].set_title('EDSR')
    ax[3].imshow(baseline[index], cmap='inferno')
    ax[3].set_title('Bilinear')
    plt.show()
    #plt.savefig('comparison.png')

index = np.random.randint(0, len(ground_truth_wind_speeds))
print(index)
plot_comparison(ground_truth_wind_speeds, lr_wind_speeds, wind_speed_fields_alt, baseline_alt, index)



355
